In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SpatialDropout1D
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
from tensorflow.keras import backend as K

# Load data
train_df = pd.read_csv('/content/drive/MyDrive/GBT_hackerton/train_df_1009.csv')
test_df = pd.read_csv('/content/drive/MyDrive/GBT_hackerton/test_df_1009.csv')

# Encode target labels
label_encoder = LabelEncoder()
train_df['label'] = label_encoder.fit_transform(train_df['??'])
num_classes = train_df['label'].nunique()

# Tokenize text data
tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_df['???'])

# Convert texts to sequences and pad them
X_train = tokenizer.texts_to_sequences(train_df['???'])
X_test = tokenizer.texts_to_sequences(test_df['???'])
max_len = 200
X_train_padded = pad_sequences(X_train, maxlen=max_len, padding='post')
X_test_padded = pad_sequences(X_test, maxlen=max_len, padding='post')

# Prepare target variable
y_train = to_categorical(train_df['label'], num_classes=num_classes)

# Define the LSTM model
model = Sequential([
    Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=128, input_length=max_len),
    SpatialDropout1D(0.2),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])

# ???
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
model.fit(X_train_padded, y_train, epochs=5, batch_size=32, validation_split=0.2)

# Make predictions on test data
predictions = model.predict(X_test_padded)
predicted_labels = label_encoder.inverse_transform(predictions.argmax(axis=1))

# Save predictions
test_df['predicted_label'] = predicted_labels
test_df[['ID', 'predicted_label']].to_csv('/content/drive/MyDrive/GBT_hackerton/yk_submission_lstm.csv', index=False)


In [ ]:
result = pd.read_csv('/content/drive/MyDrive/GBT_hackerton/yk_submission_lstm.csv')
result['predicted_label'].value_counts()

In [ ]:
result = result.rename(columns={'predicted_label':'??'})

In [ ]:
result

In [ ]:
result.to_csv('/content/drive/MyDrive/GBT_hackerton/yk_submission3.csv', index=False)